# Chronos-2: One-Step-Ahead Filtering

Walks through a user-specified time series one step at a time,
predicting the next value and then incorporating the true observation
into the context — mimicking Bayesian filtering.

Set `TS_NAME` (or `TS_INDEX`) in the configuration cell to choose a series.

In [ ]:
# ── Configuration ─────────────────────────────────────────────
TS_NAME = "LGA008EFAPRG910_cleaned"  # Column name from the CSV
TS_INDEX = None                  # If set (int), overrides TS_NAME

X_FULL_PATH = "/home/dw/cuTAGI_DW/data/hq/ts_weekly_values_final.csv"
DATES_FULL_PATH = "/home/dw/cuTAGI_DW/data/hq/ts_weekly_datetimes_final.csv"

MIN_CONTEXT = 52     # Minimum observations before first prediction
MAX_CONTEXT = None   # Rolling window size (None = growing context)

CHRONOS_MODEL = "amazon/chronos-2"
SEED = 42

In [ ]:
import ctypes
import sysconfig
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Pre-load PyTorch CUDA deps to avoid library conflicts
nvidia_root = Path(sysconfig.get_paths()["purelib"]) / "nvidia"
for _rel in (
    "nvjitlink/lib/libnvJitLink.so.12",
    "cusparse/lib/libcusparse.so.12",
):
    _lib = nvidia_root / _rel
    if _lib.exists():
        try:
            ctypes.CDLL(str(_lib), mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass

import torch
from chronos import Chronos2Pipeline

from pytagi import Normalizer as normalizer
import pytagi.metric as metric

In [ ]:
values_df = pd.read_csv(X_FULL_PATH)
dates_df = pd.read_csv(DATES_FULL_PATH)

# Select series
ts_name = values_df.columns[TS_INDEX] if TS_INDEX is not None else TS_NAME

series_values = values_df[ts_name].values.astype(np.float32)
series_dates = pd.to_datetime(dates_df[ts_name])

# Trim trailing NaNs
valid_mask = ~np.isnan(series_values)
if np.any(valid_mask):
    last_valid = np.where(valid_mask)[0][-1]
    series_values = series_values[: last_valid + 1]
    series_dates = series_dates.iloc[: last_valid + 1].reset_index(drop=True)

n = len(series_values)
print(
    f"Series: {ts_name}  |  Length: {n}  |  "
    f"{series_dates.iloc[0].date()} \u2192 {series_dates.iloc[-1].date()}  |  "
    f"interior NaNs: {np.isnan(series_values).sum()}"
)

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = Chronos2Pipeline.from_pretrained(CHRONOS_MODEL, device_map=device)
print(f"Loaded {CHRONOS_MODEL} on {device}")

In [ ]:
pred_means = np.full(n, np.nan, dtype=np.float32)
pred_q10 = np.full(n, np.nan, dtype=np.float32)
pred_q90 = np.full(n, np.nan, dtype=np.float32)

Z90 = 1.2815515655446004

for t in tqdm(range(MIN_CONTEXT, n), desc="1-step-ahead filtering"):
    ctx_start = max(0, t - MAX_CONTEXT) if MAX_CONTEXT is not None else 0
    context = series_values[ctx_start:t]
    ctx_dates = series_dates.iloc[ctx_start:t]

    context_df = pd.DataFrame({
        "id": "0",
        "timestamp": ctx_dates.values,
        "target": context,
    })

    with torch.no_grad():
        forecast_df = pipeline.predict_df(
            context_df,
            prediction_length=1,
            quantile_levels=[0.1, 0.5, 0.9],
            id_column="id",
            timestamp_column="timestamp",
            target="target",
        )

    pred_means[t] = forecast_df["0.5"].iloc[0]
    pred_q10[t] = forecast_df["0.1"].iloc[0]
    pred_q90[t] = forecast_df["0.9"].iloc[0]

pred_std = np.where(
    ~np.isnan(pred_means),
    np.maximum((pred_q90 - pred_q10) / (2 * Z90), 1e-6),
    np.nan,
).astype(np.float32)

print(f"Predictions computed for steps [{MIN_CONTEXT}, {n})")

In [ ]:
# Metrics over all predicted steps [MIN_CONTEXT, n)
pred_mask = ~np.isnan(pred_means)
all_true = series_values[pred_mask]
all_pred = pred_means[pred_mask]
all_std = pred_std[pred_mask]

# Standardize using the initial context statistics
ctx_mean = np.nanmean(series_values[:MIN_CONTEXT])
ctx_std = np.nanstd(series_values[:MIN_CONTEXT])

stand_true = normalizer.standardize(all_true, ctx_mean, ctx_std)
stand_pred = normalizer.standardize(all_pred, ctx_mean, ctx_std)
stand_s = normalizer.standardize_std(all_std, ctx_std)

res_rmse = metric.rmse(stand_pred, stand_true)
res_loglik = metric.log_likelihood(stand_pred, stand_true, stand_s)
res_mae = metric.mae(stand_pred, stand_true)
res_p50 = metric.Np50(all_true, all_pred)
res_p90 = metric.Np90(all_true, all_pred, all_std)

print(f"Filtering metrics for {ts_name}:")
print(f"  RMSE:   {res_rmse:.4f}")
print(f"  LogLik: {res_loglik:.4f}")
print(f"  MAE:    {res_mae:.4f}")
print(f"  Np50:   {res_p50:.4f}")
print(f"  Np90:   {res_p90:.4f}")

In [ ]:
import matplotlib as mpl

SINGLE_COL = (3.5, 2.5)
DOUBLE_COL = (6.5, 3.5)

mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "font.family": "serif",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": r"\usepackage{amsfonts}\usepackage{amssymb}\usepackage{amsmath}",
    "lines.linewidth": 1,
    "figure.figsize": SINGLE_COL,
    "font.size": 9,
    "savefig.dpi": 300,
})

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=DOUBLE_COL)
x = np.arange(n)

# Observations
ax.plot(x, series_values, color="tab:red", label="Observations")

# 1-step-ahead predictions
ax.plot(x, pred_means, color="tab:blue", label="Predictions")

# Epistemic uncertainty band
ax.fill_between(
    x,
    pred_means - pred_std,
    pred_means + pred_std,
    color="tab:blue",
    alpha=0.3,
    ec="none",
    label=r"$\pm 1\sigma$",
)

ax.legend(
    loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=3, frameon=False
)
ax.set_xlabel("Time step")
ax.set_ylabel("Value")

out_dir = Path("experiments/out")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "chronos_filtering.svg", bbox_inches="tight")
# fig.savefig(out_dir / f"chronos_filtering_{ts_name}.pgf", bbox_inches="tight")
plt.show()